# Maia-3 · quickstart

<a href="https://colab.research.google.com/github/chessformer-lens/chessformer_lens/blob/main/notebooks/maia_quickstart.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Look inside Maia-3, a transformer trained to play like humans at a given rating, with
[chessformer_lens](https://github.com/chessformer-lens/chessformer_lens): how its policy, its
attention and its internals change with skill. Runs on the free CPU runtime.

In [ ]:
# Maia-3's model code is not on PyPI, so it is installed from GitHub first
!pip install -q https://github.com/CSSLab/maia3/archive/1e13597c42d4858b7cfd7cfdae01e297263364b2.zip
!pip install -q git+https://github.com/chessformer-lens/chessformer_lens

## Load Maia-3
`maia3-5m` downloads from Hugging Face on first use; `maia3-23m` and `maia3-79m` are larger.

In [ ]:
import chess
from chessformer_lens import MaiaEngine
import chessformer_lens.interp_plot as ip
import chessformer_lens.interp_widget as iw

eng = MaiaEngine("maia3-5m")
ip.set_theme("light")
print(f"{eng.cfg.num_blocks} layers × {eng.cfg.num_heads} heads × {eng.cfg.dim_vit}d on {eng.device}")

## Skill changes the move
The same position at two ratings: blue is 1100, green is 2200. The Opera game (Morphy, 1858),
just before the queen sacrifice.

In [ ]:
board = chess.Board("4kb1r/p2n1ppp/4q3/4p1B1/4P3/1Q6/PPP2PPP/2KR4 w k - 0 16")
ip.plot_position(eng, board, 1100, elo_b=2200)

## When does the move form?
The logit lens reads a move's logit off the residual stream after every attention and MLP
sub-layer. Here: the strongest player's move, at both ratings.

In [ ]:
best = eng.evaluate(board, 2200)["policy"][0][0]
ip.plot_move_microscope(eng, board, 1100, best, elo_b=2200)

## Which heads carry it
Every head is ablated in turn and the change in the move's logit recorded (blue = carries the move,
orange = suppresses it). 72 forward passes on the 5M model.

In [ ]:
ip.plot_carrier_heads(eng, board, 2200, uci=best)

## Attention, live
Pick a layer and a head, click a square to set the query.

In [ ]:
iw.attention_widget(eng, board, 2200, layer=5, head=5)

## The whole app, in a new tab
Every panel above and more (the rating slider, the move microscope, neurons) in the app, on the
engine you already loaded. In Colab it opens in a new window through Colab's port proxy.

In [ ]:
from chessformer_lens import web
app = web.serve(engine=eng)